In [1]:
from pathlib import Path
import json
import pandas as pd

def build_manual_transcript_list(out_dir: str, out_name: str = "needs_manual_transcript_capture"):
    out_path = Path(out_dir)

    # Prefer using the status report if it exists (most informative)
    status_csv = out_path / "full_status_report.csv"
    rows = []

    if status_csv.exists():
        df = pd.read_csv(status_csv)

        # Keep only videos that were selected and transcript failed/unavailable
        # (Column names are from the script we used; this is defensive in case some are missing.)
        if "Selected" in df.columns:
            df = df[df["Selected"] == True]

        if "Transcript_Available" in df.columns:
            df = df[df["Transcript_Available"] == False]

        # Ensure required fields exist
        for col in ["YouTube_Video_ID", "Channel_ID", "Channel_Name", "Video_Title", "PublishedAt", "DurationSeconds",
                    "Failure_Reason", "Provider"]:
            if col not in df.columns:
                df[col] = ""

        df["Video_URL"] = df["YouTube_Video_ID"].apply(lambda v: f"https://www.youtube.com/watch?v={v}" if isinstance(v, str) and v else "")
        df["Manual_Action"] = "Open video → ⋯ / More → Show transcript → copy/paste into .txt"

        # Output columns (ordered)
        out_df = df[[
            "Channel_ID", "Channel_Name", "YouTube_Video_ID", "Video_URL",
            "Video_Title", "PublishedAt", "DurationSeconds",
            "Provider", "Failure_Reason",
            "Manual_Action"
        ]].copy()

    else:
        # Fallback: build from *.meta.json files if full_status_report.csv isn't present
        meta_files = list(out_path.rglob("V*.meta.json"))
        for mf in meta_files:
            try:
                data = json.loads(mf.read_text(encoding="utf-8"))
                vid = data.get("YouTube_Video_ID", "")
                rows.append({
                    "Channel_ID": data.get("Channel_ID", ""),
                    "Channel_Name": data.get("Channel_Name", ""),
                    "YouTube_Video_ID": vid,
                    "Video_URL": f"https://www.youtube.com/watch?v={vid}" if vid else "",
                    "Video_Title": data.get("Video_Title", ""),
                    "PublishedAt": data.get("PublishedAt", ""),
                    "DurationSeconds": data.get("DurationSeconds", ""),
                    "Provider": data.get("Transcript_Provider", ""),
                    "Failure_Reason": data.get("Transcript_Reason", ""),
                    "Manual_Action": "Open video → ⋯ / More → Show transcript → copy/paste into .txt",
                })
            except Exception:
                continue

        out_df = pd.DataFrame(rows)

        # Keep only those that don't have a transcript available (if the field exists)
        if "Transcript_Available" in out_df.columns:
            out_df = out_df[out_df["Transcript_Available"] == False]

    # De-duplicate by video id
    if "YouTube_Video_ID" in out_df.columns:
        out_df = out_df.drop_duplicates(subset=["YouTube_Video_ID"], keep="first")

    # Save
    csv_path = out_path / f"{out_name}.csv"
    xlsx_path = out_path / f"{out_name}.xlsx"

    out_df.to_csv(csv_path, index=False, encoding="utf-8")
    try:
        out_df.to_excel(xlsx_path, index=False)
    except Exception:
        xlsx_path = None

    return out_df, csv_path, xlsx_path



In [2]:
# --- Run it on your folder ---
manual_df, csv_path, xlsx_path = build_manual_transcript_list(
    out_dir="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts"
)

print("Saved:", csv_path)
if xlsx_path:
    print("Saved:", xlsx_path)

manual_df.head(20)

Saved: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts/needs_manual_transcript_capture.csv
Saved: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts/needs_manual_transcript_capture.xlsx


""
